[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C09_Reasoning_TTC_Course/02_self_consistency/02_self_consistency_bon.ipynb)

# 02 · Self-Consistency 与 Best-of-N：采样聚合的统计学

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy/matplotlib 模拟，自包含，无需模型与 API key。

**本 notebook 你将完成：**

1. 构造**答案分布生成器**：每道题一个离散答案分布（正确答案概率 $q$ + 错误答案"分散/集中"两种形态）；
2. 实现 `majority_vote` 并画 **accuracy vs N** 曲线——陪审团定理的两面：$q>w_{\max}$ 时指数收敛到 1，反之收敛到 0；
3. 实现 **pass@k 无偏估计**（Chen 2021）并与暴力模拟对照，亲眼看到插件估计的偏差；
4. 复现 **Large Language Monkeys** 式 coverage vs k 幂律曲线（Beta 难度分布 → log-log 直线）；
5. **BoN 三种选择器**对比：oracle vs majority vs random；
6. 用一个**温度旋钮**控制答案分布的熵，画聚合收益 vs 温度的**倒 U**；
7. 4 道 ✏️ 练习巩固核心估计量。

参考：[Wang 2022] *Self-Consistency Improves Chain of Thought Reasoning* (arXiv:2203.11171)、[Chen 2021] *Evaluating LLMs Trained on Code* (arXiv:2107.03374)、[Brown 2024] *Large Language Monkeys* (arXiv:2407.21787)。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

def make_dist(q, n_wrong=9, concentrated=False, share=0.8):
    """构造一道题的答案分布（index 0 = 正确答案，概率 q）。
    concentrated=False: 1-q 在 n_wrong 个错误答案上均匀分散（diverse errors）
    concentrated=True : 1-q 的 share 比例集中在同一个错误答案上（systematic bias）"""
    if concentrated:
        wrong = np.full(n_wrong, (1 - q) * (1 - share) / (n_wrong - 1))
        wrong[0] = (1 - q) * share
    else:
        wrong = np.full(n_wrong, (1 - q) / n_wrong)
    return np.concatenate([[q], wrong])

def sample_answers(dist, N, rng):
    """模拟采 N 条最终答案（answer id；0 = 正确答案）"""
    return rng.choice(len(dist), size=N, p=dist)

# 同样的 q=0.35，两种命运完全相反的分布形态
d_diverse = make_dist(0.35, concentrated=False)   # 峰在正确答案：0.35 > 0.072
d_biased  = make_dist(0.35, concentrated=True)    # 峰在错误答案：0.35 < 0.52
for name, d in [("diverse 错误分散", d_diverse), ("biased  系统偏差", d_biased)]:
    print(f"{name}: q={d[0]:.2f}  w_max={d[1:].max():.3f}  "
          f"12 条样本: {sample_answers(d, 12, rng)}")


## 1 · Majority vote 与陪审团定理的两面

Self-consistency（Wang 2022）= 采 $N$ 条 CoT，对**最终答案**做相对多数表决（plurality vote）。
$N$ 条样本中答案 $a$ 的得票 $C_N(a)\sim \mathrm{Binomial}(N, p(a))$，大数定律给出极限行为：

$$ q > w_{\max} \;\Rightarrow\; P_{\text{maj}}(N) \to 1, \qquad q < w_{\max} \;\Rightarrow\; P_{\text{maj}}(N) \to 0 $$

其中 $w_{\max}$ 是最大的单个错误答案概率。注意正确答案**不需要过半**，只需是众数——
错误分散时 $q=0.35$ 足够；错误集中时同样的 $q$ 会被投票**放大成自信的错误**。
收敛速率是指数的（Chernoff：margin $q-w_{\max}$ 的平方进指数），margin 小则收敛极慢。

In [ ]:
def majority_vote(samples):
    """众数答案；np.bincount.argmax 平票时取最小 id（练习 1 实现更讲究的平票规则）"""
    return int(np.bincount(samples).argmax())

print("demo:", majority_vote(np.array([4, 2, 4, 7, 4])), "(应为 4)")

def majority_acc(dist, N, trials, rng):
    """Monte Carlo 估计 majority@N 的正确率（向量化：一次采 trials×N）"""
    K = len(dist)
    s = rng.choice(K, size=(trials, N), p=dist)
    counts = (s[:, :, None] == np.arange(K)).sum(axis=1)      # (trials, K) 计票
    return (counts.argmax(axis=1) == 0).mean()

d_hard = make_dist(0.12, concentrated=False)   # q=0.12 vs w=0.098：margin 极小
Ns = [1, 3, 5, 9, 15, 25, 41, 61, 101, 151, 201]
curves = {}
for name, d in [("diverse: q=.35 > w=.07", d_diverse),
                ("biased : q=.35 < w=.52", d_biased),
                ("hard   : q=.12 ≈ w=.10", d_hard)]:
    curves[name] = [majority_acc(d, N, trials=3000, rng=rng) for N in Ns]

plt.figure(figsize=(7, 4.2))
for name, ys in curves.items():
    plt.plot(Ns, ys, marker="o", ms=4, label=name)
plt.axhline(0.35, color="gray", ls=":", lw=1, label="pass@1 = q = 0.35")
plt.xscale("log"); plt.xlabel("N (samples)"); plt.ylabel("majority@N accuracy")
plt.title("Jury theorem, both ways: voting amplifies the mode")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for name, ys in curves.items():
    print(f"{name}:  N=1 → {ys[0]:.3f}   N=201 → {ys[-1]:.3f}")


三条曲线 = 三种命运：错误分散 → 升向 1；系统性偏差 → **降向 0**（投票把 52% 的共享错误投成必然）；
margin 极小 → 缓慢爬升（Chernoff 速率被 $(q-w_{\max})^2$ 平方惩罚）。
**majority@N 上不去 ≠ 模型不会做**，可能只是众数错了——这正是接下来 pass@k 要回答的问题。

## 2 · pass@k 的无偏估计（Chen 2021, arXiv:2107.03374）

pass@k：采 $k$ 条、**任意一条**正确即算对（需要 oracle 判分），测的是**覆盖率/能力上限**。
单题真值 $\text{pass@}k = 1-(1-q)^k$。估计时对每题采 $n\ge k$ 条、数出 $c$ 条正确：

$$ \widehat{\text{pass@}k} = 1 - \binom{n-c}{k}\Big/\binom{n}{k} $$

这是无偏估计（超几何恒等式）；而插件估计 $1-(1-c/n)^k$ 是**有偏的**：
$f(q)=1-(1-q)^k$ 凹 → Jensen 给出 $\mathbb{E}[f(\hat q)]\le f(q)$，**系统性低估**覆盖率。
数值实现用连乘 $\binom{n-c}{k}/\binom{n}{k} = \prod_{i=n-c+1}^{n}(1-k/i)$，绝不直接算大组合数。

In [ ]:
from math import comb

def pass_at_k(n, c, k):
    """Chen 2021 无偏估计：1 - C(n-c,k)/C(n,k)，连乘形式避免大组合数"""
    if n - c < k:
        return 1.0
    return float(1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))

# ① 与组合数定义逐点一致
for (n, c, k) in [(10, 3, 2), (50, 7, 10), (20, 0, 5), (20, 18, 5)]:
    ref = 1.0 if n - c < k else 1 - comb(n - c, k) / comb(n, k)
    assert abs(pass_at_k(n, c, k) - ref) < 1e-12
print("① 与组合数定义一致 OK")

# ② 与暴力子抽样模拟对照：固定 n=50 条样本中 c=7 条正确，反复无放回抽 k=10 条
n, c, k = 50, 7, 10
flags = np.array([1] * c + [0] * (n - c))
hits = np.mean([flags[rng.choice(n, size=k, replace=False)].any()
                for _ in range(20000)])
print(f"② 暴力模拟 {hits:.4f}  vs  闭式 {pass_at_k(n, c, k):.4f}")
assert abs(hits - pass_at_k(n, c, k)) < 0.01

# ③ 无偏 vs 插件：真值 q=0.12，反复观测 c~Binomial(n,q)
q, n, k = 0.12, 50, 10
true_val = 1 - (1 - q) ** k
cs = rng.binomial(n, q, size=4000)
est_unbiased = np.mean([pass_at_k(n, c, k) for c in cs])
est_plugin   = np.mean([1 - (1 - c / n) ** k for c in cs])
print(f"③ 真值 {true_val:.4f} | 无偏估计均值 {est_unbiased:.4f} | "
      f"插件估计均值 {est_plugin:.4f} ← 凹函数 + Jensen：系统性低估")
assert abs(est_unbiased - true_val) < 0.01 and est_plugin < true_val - 0.01


## 3 · Large Language Monkeys：coverage vs k 的幂律（Brown 2024）

单题覆盖率 $1-(1-q_i)^k$ 是**指数饱和**的，毫无幂律。幂律来自**对题目难度求平均**：

$$ c_k = 1 - \mathbb{E}_{q\sim f}\big[(1-q)^k\big] $$

若难度分布 $f$ 在 $q\to 0^+$ 有 $q^{\alpha-1}$ 形密度（大量"几乎不会做"的长尾难题，如 $\mathrm{Beta}(\alpha,\beta)$、$\alpha<1$），
则 $1-c_k \sim k^{-\alpha}$——**幂律是难题长尾的集体签名**，拟合指数度量的是 benchmark 难度分布的尾部厚度。
下面用 $\mathrm{Beta}(0.25, 3)$ 难度直接复现 log-log 直线，并验证斜率 $\approx -\alpha$。

In [ ]:
ALPHA = 0.25
M = 4000                                   # 题目数
q_i = rng.beta(ALPHA, 3.0, size=M)         # 每题一个"单样本正确率"
ks = np.unique(np.logspace(0, 3, 25).astype(int))   # k: 1 → 1000

coverage = np.array([(1 - (1 - q_i) ** k).mean() for k in ks])   # 解析平均
# Monte Carlo 对照：每题采 n=2000 条，数 c_i，用无偏估计器聚合
n = 2000
c_i = rng.binomial(n, q_i)
cov_mc = np.array([np.mean([pass_at_k(n, c, k) for c in c_i]) for k in ks])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(ks, coverage, "-o", ms=3, label="analytic $1-E[(1-q)^k]$")
axes[0].plot(ks, cov_mc, "x", ms=5, label="unbiased estimator (n=2000)")
axes[0].set_xscale("log"); axes[0].set_xlabel("k"); axes[0].set_ylabel("coverage (pass@k)")
axes[0].set_title("Monkeys-style coverage curve"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

fail = 1 - coverage
mask = ks >= 10
slope, intercept = np.polyfit(np.log(ks[mask]), np.log(fail[mask]), 1)
axes[1].loglog(ks, fail, "-o", ms=3, label="$1-c_k$")
axes[1].loglog(ks, np.exp(intercept) * ks ** slope, "--",
               label=f"power-law fit, slope={slope:.3f}")
axes[1].set_xlabel("k"); axes[1].set_ylabel("1 - coverage")
axes[1].set_title(f"log-log line, theory slope = -{ALPHA}")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

print(f"k=1 coverage = {coverage[0]:.3f}  →  k=1000 coverage = {coverage[-1]:.3f}")
print(f"幂律拟合斜率 = {slope:.3f}（理论 ≈ -{ALPHA}）")
assert slope < 0 and abs(cov_mc - coverage).max() < 0.02


暴力采样把 coverage 从 ~0.18 推到 ~0.8——这就是 Monkeys 的"惊人收益"。但 coverage 是 **oracle 口径**：
答案在 1000 条样本里 ≠ 你能把它挑出来。下面比较三种**选择器**，再看温度如何同时拨动质量与多样性。

## 4 · BoN 选择器对比 + 温度的倒 U

- **oracle**：上帝视角挑（= pass@N，上界）；**majority**：投票（无需 verifier）；**random**：随机挑一条（= pass@1 期望，下界）。
- **温度**：$p_T(a)\propto e^{z_a/T}$。构造一道"贪心答案错误"的题（$z_{\text{wrong}}>z_{\text{correct}}$）：
  $T$ 太低 → 所有样本都是同一个错误答案，聚合零收益；$T$ 太高 → 均匀噪声；中间存在最优 $T^\star$ → **倒 U**。

In [ ]:
dist = make_dist(0.25, n_wrong=9, concentrated=False)   # q=0.25 > w=0.083
Ns = [1, 2, 4, 8, 16, 32, 64, 128]
trials, K = 4000, len(dist)
acc = {"oracle BoN (= pass@N)": [], "majority vote": [], "random pick": []}
for N in Ns:
    s = rng.choice(K, size=(trials, N), p=dist)
    counts = (s[:, :, None] == np.arange(K)).sum(axis=1)
    acc["oracle BoN (= pass@N)"].append((s == 0).any(axis=1).mean())
    acc["majority vote"].append((counts.argmax(axis=1) == 0).mean())
    acc["random pick"].append((s[:, 0] == 0).mean())

plt.figure(figsize=(7, 4))
for name, ys in acc.items():
    plt.plot(Ns, ys, marker="o", ms=4, label=name)
plt.xscale("log"); plt.xlabel("N"); plt.ylabel("accuracy")
plt.title("BoN selectors (q=0.25, diverse errors): oracle ≥ majority ≥ random")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for name, ys in acc.items():
    print(f"{name:24s} N=1 → {ys[0]:.3f}   N=128 → {ys[-1]:.3f}")


In [ ]:
# 温度旋钮：一道"贪心答案错误"的题。z[0]=正确答案 logit，z[1]=更高的错误 logit
K = 30
z = np.concatenate([[1.2, 1.6], rng.normal(0.0, 0.4, size=K - 2)])
assert z.argmax() == 1     # 贪心解码（T→0）必然选中错误答案

def softmax_T(z, T):
    a = (z - z.max()) / T
    e = np.exp(a)
    return e / e.sum()

Ts = np.linspace(0.05, 4.0, 30)
N, trials = 16, 2000
p1, pN, majN = [], [], []
for T in Ts:
    p = softmax_T(z, T)
    q = p[0]
    p1.append(q)                       # pass@1 = q(T)
    pN.append(1 - (1 - q) ** N)        # pass@16（解析）
    s = rng.choice(K, size=(trials, N), p=p)
    counts = (s[:, :, None] == np.arange(K)).sum(axis=1)
    majN.append((counts.argmax(axis=1) == 0).mean())
p1, pN, majN = map(np.array, (p1, pN, majN))

plt.figure(figsize=(7, 4))
plt.plot(Ts, p1, label="pass@1 = q(T)")
plt.plot(Ts, pN, label="pass@16（coverage）")
plt.plot(Ts, majN, label="majority@16")
plt.plot(Ts, pN - p1, "--", label="gain: pass@16 − pass@1")
plt.xlabel("temperature T"); plt.ylabel("accuracy")
plt.title("Diversity-quality tradeoff: inverted-U of aggregation gain")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

T_star = Ts[np.argmax(pN)]
print(f"pass@16 峰值温度 T* = {T_star:.2f}（T→0 退化为贪心错误答案，T→∞ 趋于均匀 1/{K}）")
print(f"majority@16 全程被压制：w_max(T) > q(T) 对一切 T 成立（softmax 保序）——")
print("系统性偏差投票救不了，只有 oracle/verifier 能兑现覆盖率。")
assert pN.max() > 0.6 and pN[0] < 0.1 and pN[-1] < pN.max()   # 倒 U 成立


---
## ✏️ 练习 1：实现 `majority_vote_ex`（含平票规则）

实现 `majority_vote_ex(samples, tie="first", rng=None)`：返回得票最多的答案 id。
平票时：`tie="first"` 返回平票者中**在 samples 里最先出现**的那个；`tie="random"` 用传入的 `rng` 在平票者中均匀随机挑一个。

**提示**：`np.bincount` 计票 → `np.flatnonzero(counts == counts.max())` 找平票集合；
"最先出现"用 `np.argmax(samples == a)` 找首次位置取最小。约 12 行；
边界：单元素序列、无平票时 tie 参数不应影响结果。

In [ ]:
def majority_vote_ex(samples, tie="first", rng=None):
    # TODO:
    #  1) samples 转 np.array，np.bincount 计票
    #  2) 找出得票最高的答案集合 leaders
    #  3) 只有一个 leader 直接返回
    #  4) tie=="random": rng.choice(leaders)
    #     tie=="first" : 返回在 samples 中首次出现位置最早的 leader
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
rng_t = np.random.default_rng(0)
assert majority_vote_ex([1, 2, 2, 3]) == 2                       # 无平票
assert majority_vote_ex([7]) == 7                                # 边界：单元素
assert majority_vote_ex([3, 1, 1, 3], tie="first") == 3          # 平票：3 先出现
assert majority_vote_ex([5, 4, 4, 5, 6], tie="first") == 5       # 平票：5 先出现
picks = np.array([majority_vote_ex([0, 1], tie="random", rng=rng_t)
                  for _ in range(2000)])
assert set(picks) == {0, 1} and 0.42 < (picks == 0).mean() < 0.58   # 随机平票 ≈ 各 50%
print("✅ 练习 1 通过")


## ✏️ 练习 2：实现 `pass_at_k_ex`（log 域防溢出）

实现 Chen 2021 无偏估计 `pass_at_k_ex(n, c, k)`，要求**在对数域累加**：
$\log\frac{\binom{n-c}{k}}{\binom{n}{k}} = \sum_{i=n-c+1}^{n}\log(1-k/i)$，最后 `1 - exp(...)`，
使 $n=10^6$ 这类量级不溢出。

**提示**：边界先行——`n-c < k` 返回 1.0、`c == 0` 返回 0.0；
主体 3 行：`i = np.arange(n-c+1, n+1)` → `np.log1p(-k/i)` 求和 → `1 - np.exp(...)`。
注意求和长度是 $c$ 而不是 $n$，所以大 $n$ 也不慢。

In [ ]:
def pass_at_k_ex(n, c, k):
    # TODO:
    #  1) 边界：n - c < k -> 1.0；c == 0 -> 0.0
    #  2) i = np.arange(n - c + 1, n + 1)
    #  3) return 1 - exp( sum( log1p(-k / i) ) )
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
from math import comb as _comb
for _n in [5, 12, 30]:                                     # ① 与组合数定义一致
    for _c in range(_n + 1):
        for _k in range(1, _n + 1):
            ref = 1.0 if _n - _c < _k else 1 - _comb(_n - _c, _k) / _comb(_n, _k)
            assert abs(pass_at_k_ex(_n, _c, _k) - ref) < 1e-9, (_n, _c, _k)
assert pass_at_k_ex(100, 0, 10) == 0.0                     # ② 边界
assert pass_at_k_ex(100, 95, 10) == 1.0
vals = [pass_at_k_ex(50, 5, k) for k in range(1, 46)]      # ③ 对 k 单调不减
assert all(b >= a - 1e-12 for a, b in zip(vals, vals[1:]))
big = pass_at_k_ex(10**6, 10, 10**5)                       # ④ 大 n 不溢出
assert 0.0 < big < 1.0 and abs(pass_at_k_ex(10**6, 1, 10**5) - 0.1) < 1e-6
print("✅ 练习 2 通过")


## ✏️ 练习 3：实现 `coverage_curve`

实现 `coverage_curve(qs, ks)`：给定每题单样本正确率数组 `qs` 与采样数列表 `ks`，
返回 `np.array`，第 $j$ 个元素为 $c_{k_j} = \frac{1}{M}\sum_i \big(1-(1-q_i)^{k_j}\big)$（解析覆盖率，无需模拟）。

**提示**：一行列表推导即可；自测会用 $\mathrm{Beta}(0.25, 3)$ 难度分布检查
$\log(1-c_k)$ vs $\log k$ 的**幂律拟合斜率为负**且量级接近 $-\alpha$。

In [ ]:
def coverage_curve(qs, ks):
    # TODO: 对每个 k 计算 mean(1 - (1-qs)**k)，返回 np.array
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
rng_t = np.random.default_rng(0)
cov = coverage_curve(np.array([0.5, 0.1]), [1, 2])         # ① 手算对照
assert np.allclose(cov, [(0.5 + 0.1) / 2, (0.75 + 0.19) / 2])
qs_t = rng_t.beta(0.25, 3.0, size=5000)
ks_t = np.unique(np.logspace(0, 3, 20).astype(int))
cov = coverage_curve(qs_t, ks_t)
assert cov.shape == (len(ks_t),)
assert np.all(np.diff(cov) >= -1e-12)                      # ② 对 k 单调不减
assert np.all((cov >= 0) & (cov <= 1))
m = ks_t >= 10                                             # ③ 幂律：log-log 斜率为负
slope = np.polyfit(np.log(ks_t[m]), np.log(1 - cov[m]), 1)[0]
assert slope < -0.05, f"斜率应为负，得到 {slope:.3f}"
assert -0.6 < slope < -0.1                                 # 量级 ≈ -α = -0.25
print(f"幂律拟合斜率 = {slope:.3f}（理论 ≈ -0.25）")
print("✅ 练习 3 通过")


## ✏️ 练习 4：实现 `expected_majority_acc`（精确计算）

实现 `expected_majority_acc(dist, N)`：majority@N 正确率的**精确值**（不靠模拟）。
枚举所有计票向量 $(c_0,\dots,c_{K-1})$（$\sum c_i = N$），按多项分布
$P(\mathbf{c}) = \frac{N!}{\prod c_i!}\prod p_i^{c_i}$ 加权；正确答案严格领先记 1 分，
与 $m$ 个答案并列最高记 $1/m$ 分（平票均匀随机）。

**提示**：递归生成器枚举 compositions（`K=1` 时 yield `(n,)`，否则对首位取值递归），
$K$ 个答案、$N$ 条样本共 $\binom{N+K-1}{K-1}$ 项，测试里 $K\le 3, N\le 9$ 很小；
`math.factorial` 算多项式系数即可。约 20 行。自测含两个手算闭式 + 与多项分布 Monte Carlo 对照。

In [ ]:
def expected_majority_acc(dist, N):
    # TODO:
    #  1) 写一个 compositions(n, k) 生成器：所有非负整数 k 元组、和为 n
    #  2) 对每个计票向量 cvec 计算多项分布概率
    #  3) m = max(cvec)；若 c_0 == m：acc += prob / (并列最高的个数)
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
rng_t = np.random.default_rng(0)
# ① 二元 N=3 闭式：p^3 + 3 p^2 (1-p) = 0.784 (p=0.7)
assert abs(expected_majority_acc([0.7, 0.3], 3) - 0.784) < 1e-9
# ② 对称平票：p=0.5, N=2 -> 0.25 + 0.5*0.5 = 0.5
assert abs(expected_majority_acc([0.5, 0.5], 2) - 0.5) < 1e-9
# ③ N=1 退化为 q
assert abs(expected_majority_acc([0.4, 0.3, 0.3], 1) - 0.4) < 1e-9
# ④ 与 Monte Carlo（多项分布 + 平票随机）一致
dist_t = np.array([0.4, 0.3, 0.3])
counts = rng_t.multinomial(5, dist_t, size=40000)
mx = counts.max(axis=1)
lead = counts == mx[:, None]
mc = np.where(lead[:, 0], 1.0 / lead.sum(axis=1), 0.0).mean()
assert abs(expected_majority_acc(dist_t, 5) - mc) < 0.02
# ⑤ q > w_max 时 N 越大越准（陪审团定理）
assert expected_majority_acc(dist_t, 9) > expected_majority_acc(dist_t, 1)
print("✅ 练习 4 通过")


---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def majority_vote_ex(samples, tie="first", rng=None):
    samples = np.asarray(samples)
    counts = np.bincount(samples)
    leaders = np.flatnonzero(counts == counts.max())
    if len(leaders) == 1:
        return int(leaders[0])
    if tie == "random":
        return int(rng.choice(leaders))
    # tie == "first": 取在 samples 中首次出现位置最早的 leader
    first_pos = [int(np.argmax(samples == a)) for a in leaders]
    return int(leaders[int(np.argmin(first_pos))])


In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def pass_at_k_ex(n, c, k):
    if n - c < k:
        return 1.0
    if c == 0:
        return 0.0
    i = np.arange(n - c + 1, n + 1, dtype=float)   # 长度 c，与 n 无关
    return float(1.0 - np.exp(np.sum(np.log1p(-k / i))))


In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def coverage_curve(qs, ks):
    qs = np.asarray(qs, dtype=float)
    return np.array([np.mean(1.0 - (1.0 - qs) ** k) for k in ks])


In [ ]:
# 练习 4 参考答案（先自己做，再对照）
from math import factorial

def expected_majority_acc(dist, N):
    dist = np.asarray(dist, dtype=float)
    K = len(dist)

    def compositions(n, k):
        if k == 1:
            yield (n,)
            return
        for first in range(n + 1):
            for rest in compositions(n - first, k - 1):
                yield (first,) + rest

    acc = 0.0
    for cvec in compositions(N, K):
        prob = factorial(N)
        for i, c in enumerate(cvec):
            prob = prob / factorial(c) * dist[i] ** c
        m = max(cvec)
        if cvec[0] == m:
            acc += prob / sum(1 for c in cvec if c == m)   # 平票均匀随机
    return acc


---
## 🎯 真实数据胶囊题：真实 GSM8K 上的 self-consistency（Condorcet 增益）

对同题多次采样取多数答案。若单次正确率 > 0.5，多数投票随采样数提升（Condorcet 陪审团定理）。用真实 GSM8K 金标模拟带噪采样，验证多数投票准确率随 k 上升。

> 本模块新增的**真实数据**练习：自包含、用真实 GSM8K 把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.reasoning_ttc_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def steps(a): return max(1, a.count("<<"))   # 真实推理步数代理

rows=gsm8k(150); golds=[gold(r["answer"]) for r in rows]
rng=np.random.default_rng(0)
def sample(g, p=0.55):
    return g if rng.random()<p else str(int(rng.integers(0,99999)))

**练习**：实现 `self_consistency_acc(golds, k, p, seed)`：每题采 k 次取多数答案，返回准确率。

In [ ]:
def self_consistency_acc(golds, k, p=0.55, seed=0):
    # TODO: 每题采 k 次(用 sample)，多数投票，与 gold 比，返回平均准确率
    raise NotImplementedError


In [ ]:
# 自测
a1=self_consistency_acc(golds,1,0.55,1)
a9=self_consistency_acc(golds,9,0.55,1)
assert a9 > a1, "k 越大多数投票越准(Condorcet, p>0.5)"
print(f"self-consistency: k=1 {a1:.2f} -> k=9 {a9:.2f} ✓")


### 📖 参考答案

In [ ]:
def self_consistency_acc(golds, k, p=0.55, seed=0):
    from collections import Counter
    rng2=np.random.default_rng(seed)
    def s(g): return g if rng2.random()<p else str(int(rng2.integers(0,99999)))
    c=0
    for g in golds:
        votes=[s(g) for _ in range(k)]
        if Counter(votes).most_common(1)[0][0]==g: c+=1
    return c/len(golds)
print("✓ 单次>50%时，多数投票随样本数趋近 100% (Condorcet)")